# 02 - Chunking

Current baseline: structure-aware chunking (header split, then recursive token
split for oversized sections), 500 tokens, ~100 token (20%) overlap. Tables kept
whole, never split.

Planned experiments: chunk size (300/500/800), overlap (10%/20%), structure-aware
vs naive fixed-window / recursive baseline.

## Step 1: Setup

Imports, load `data/processed/ingested.json` from `01_ingestion`, and constants
(chunk size, overlap, a token-counting function).

In [1]:
import json
from pathlib import Path

INGESTED_PATH = Path("../data/processed/ingested.json")
CHUNKS_PATH = Path("../data/processed/chunks.json")

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100


def count_tokens(text: str) -> int:
    return len(text) // 4


documents = json.loads(INGESTED_PATH.read_text(encoding="utf-8"))
len(documents)

5

## Step 2: Split by headers

Split each document's markdown `text` at heading boundaries (`##`, `###`), keeping
the heading path (e.g. `Overtime pay > How overtime pay is calculated`) attached to
each resulting section.

In [2]:
import re

# Allow leading whitespace before the '#' -- some source pages (e.g. medical-insurance)
# have headings indented by stray spaces from trafilatura extraction, which a
# strictly-anchored "^#" regex would silently miss and fold into the prior section.
HEADING_RE = re.compile(r"^\s*(#{2,3})\s+(.*)$")


def split_by_headers(text: str) -> list[dict]:
    lines = text.split("\n")
    sections = []
    stack = []  # list of (level, title)
    current_lines = []

    def flush():
        body = "\n".join(current_lines).strip()
        if body:
            heading_path = " > ".join(title for _, title in stack)
            sections.append({"heading_path": heading_path, "text": body})

    for line in lines:
        match = HEADING_RE.match(line)
        if match:
            flush()
            current_lines = []
            level = len(match.group(1))
            title = match.group(2).strip()
            while stack and stack[-1][0] >= level:
                stack.pop()
            stack.append((level, title))
        else:
            current_lines.append(line)
    flush()
    return sections


sections_by_doc = {doc["document_id"]: split_by_headers(doc["text"]) for doc in documents}
{doc_id: len(sections) for doc_id, sections in sections_by_doc.items()}

{'https-www-mom-gov-sg-employment-practices-salary-paying-salary': 7,
 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days': 14,
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions': 3,
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-medical-insurance': 4,
 'https-www-mom-gov-sg-contact-us': 1}

## Step 3: Merge undersized sections

Header splitting alone leaves some sections far too small (e.g. a 2-sentence
section). Merge adjacent small sections within the same document forward until
they approach the target chunk size, instead of emitting tiny fragments.

In [3]:
def merge_sections(sections: list[dict], chunk_size: int) -> list[dict]:
    merged = []
    buffer_texts = []
    buffer_heading_path = None

    def flush():
        if buffer_texts:
            merged.append(
                {
                    "heading_path": buffer_heading_path,
                    "text": "\n\n".join(buffer_texts),
                }
            )

    for section in sections:
        if not buffer_texts:
            buffer_texts = [section["text"]]
            buffer_heading_path = section["heading_path"]
            continue

        combined_tokens = count_tokens("\n\n".join(buffer_texts)) + count_tokens(section["text"])
        if combined_tokens <= chunk_size:
            buffer_texts.append(section["text"])
        else:
            flush()
            buffer_texts = [section["text"]]
            buffer_heading_path = section["heading_path"]

    flush()
    return merged


merged_sections_by_doc = {
    doc_id: merge_sections(sections, CHUNK_SIZE) for doc_id, sections in sections_by_doc.items()
}
{doc_id: len(sections) for doc_id, sections in merged_sections_by_doc.items()}

{'https-www-mom-gov-sg-employment-practices-salary-paying-salary': 3,
 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days': 5,
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions': 3,
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-medical-insurance': 3,
 'https-www-mom-gov-sg-contact-us': 1}

## Step 4: Sub-split oversized sections

For any (merged) section still over the max token size, recursively split it
(paragraph → sentence → word boundaries) with overlap — but never at a point that
falls inside a markdown table block. Tables stay whole even if that means a chunk
exceeds the token target.

In [5]:
TABLE_BLOCK_RE = re.compile(r"(?:[ \t]*\|.*\|[ \t]*\n(?:[ \t]*\n)*)+")


def protect_tables(text: str) -> tuple[str, dict[str, str]]:
    placeholders = {}

    def replace(match):
        key = f"\x00TABLE{len(placeholders)}\x00"
        placeholders[key] = match.group(0)
        return key

    return TABLE_BLOCK_RE.sub(replace, text), placeholders


def split_into_blocks(text: str) -> list[str]:
    protected, placeholders = protect_tables(text)
    raw_blocks = re.split(r"\n\s*\n", protected)
    return [placeholders.get(b.strip(), b.strip()) for b in raw_blocks if b.strip()]


def pack_blocks(blocks: list[str], chunk_size: int, overlap: int) -> list[str]:
    chunks = []
    current = []
    current_tokens = 0

    for block in blocks:
        block_tokens = count_tokens(block)
        if current and current_tokens + block_tokens > chunk_size:
            chunks.append("\n\n".join(current))

            overlap_blocks = []
            overlap_tokens = 0
            for b in reversed(current):
                b_tokens = count_tokens(b)
                if overlap_tokens + b_tokens > overlap:
                    break
                overlap_blocks.insert(0, b)
                overlap_tokens += b_tokens
            current, current_tokens = overlap_blocks, overlap_tokens

        current.append(block)
        current_tokens += block_tokens

    if current:
        chunks.append("\n\n".join(current))

    return chunks


def split_oversized(section: dict, chunk_size: int, overlap: int) -> list[dict]:
    if count_tokens(section["text"]) <= chunk_size:
        return [section]

    blocks = split_into_blocks(section["text"])
    sub_texts = pack_blocks(blocks, chunk_size, overlap)
    return [{"heading_path": section["heading_path"], "text": t} for t in sub_texts]


split_sections_by_doc = {
    doc_id: [
        sub for section in sections for sub in split_oversized(section, CHUNK_SIZE, CHUNK_OVERLAP)
    ]
    for doc_id, sections in merged_sections_by_doc.items()
}
{doc_id: len(sections) for doc_id, sections in split_sections_by_doc.items()}

{'https-www-mom-gov-sg-employment-practices-salary-paying-salary': 3,
 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days': 5,
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions': 4,
 'https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-medical-insurance': 3,
 'https-www-mom-gov-sg-contact-us': 1}

## Step 5: Prepend context breadcrumb

Prefix each chunk's text with `{document title} > {heading path}` before it's used
for embedding, so a standalone chunk doesn't lose the page/section context it needs
to be understood on its own.

In [6]:
titles_by_doc = {doc["document_id"]: doc["title"] for doc in documents}


def add_breadcrumb(section: dict, title: str) -> dict:
    breadcrumb = title if not section["heading_path"] else f"{title} > {section['heading_path']}"
    return {
        "heading_path": section["heading_path"],
        "text": f"{breadcrumb}\n\n{section['text']}",
    }


breadcrumbed_sections_by_doc = {
    doc_id: [add_breadcrumb(section, titles_by_doc[doc_id]) for section in sections]
    for doc_id, sections in split_sections_by_doc.items()
}

first_doc_id = next(iter(breadcrumbed_sections_by_doc))
print(breadcrumbed_sections_by_doc[first_doc_id][0]["text"][:200])

Paying salary

In accordance with the Employment Act, your employer must pay your salary at least once a month and within 7 days after the end of the salary period. There are exceptions for overtime, 


## Step 6: Assemble chunk records

One record per chunk: `{chunk_id, document_id, url, heading_path, text,
token_count}`. This is the contract `03_embedding` will consume.

In [7]:
urls_by_doc = {doc["document_id"]: doc["url"] for doc in documents}

chunks = []
for doc_id, sections in breadcrumbed_sections_by_doc.items():
    for i, section in enumerate(sections):
        chunks.append(
            {
                "chunk_id": f"{doc_id}::chunk-{i}",
                "document_id": doc_id,
                "url": urls_by_doc[doc_id],
                "heading_path": section["heading_path"],
                "text": section["text"],
                "token_count": count_tokens(section["text"]),
            }
        )

len(chunks)

16

## Step 7: Save chunked output

Write the assembled chunk records to `data/processed/chunks.json` (gitignored).

In [8]:
CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)
CHUNKS_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

CHUNKS_PATH

WindowsPath('../data/processed/chunks.json')

## Step 8: Sanity checks

Confirm no chunk wildly exceeds the token target, no table got split across two
chunks, and spot-check a few chunks read sensibly on their own.

In [9]:
def count_tables(text: str) -> int:
    return len(TABLE_BLOCK_RE.findall(text))


for doc in documents:
    doc_id = doc["document_id"]
    original_table_count = count_tables(doc["text"])
    doc_chunks = [c for c in chunks if c["document_id"] == doc_id]
    chunked_table_count = sum(count_tables(c["text"]) for c in doc_chunks)
    assert chunked_table_count == original_table_count, (
        f"Table count mismatch for {doc_id}: {original_table_count} in source vs "
        f"{chunked_table_count} across chunks — a table may have been split."
    )

for chunk in chunks:
    flag = " [over target]" if chunk["token_count"] > CHUNK_SIZE * 1.5 else ""
    print(f"{chunk['chunk_id']} — {chunk['token_count']} tokens{flag}")

print(f"\n{len(chunks)} chunks across {len(documents)} documents. All checks passed.")

https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-0 — 288 tokens
https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1 — 494 tokens
https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-2 — 182 tokens
https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-0 — 240 tokens
https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-1 — 409 tokens
https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2 — 469 tokens
https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-3 — 434 tokens
https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-4 — 152 tokens
https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-permit-conditions::chunk-0 — 127 tokens
https-www-mom-gov-sg-passes-and-permits-work-permit-for-foreign-worker-sector-specific-rules-work-perm